【チャリまも】
原田琉太、神郡正稀、杉浦琉生

1．使用したデータと方法
千葉県警が2025年分として公開した自転車盗CSV（bike.csv、10,054件）を使いました。`for` で地域・施錠・年齢を集計し、ランキングとグラフにしました。

出典： 千葉県警察・犯罪発生情報

※2025年分のCSVには過去の発生記録も含まれます。件数はこのCSVに収録された件数です。

※レポートは bike.csv のみを集計。チャットボットは2018～2025年の資料を使い、同じCSVは重複集計しません。

In [4]:
# 2．CSVの読み込み（標準ライブラリだけで実行できます）
import csv
from pathlib import Path
from html import escape
from IPython.display import display, HTML

ファイル名 = Path("bike.csv")

if not ファイル名.is_file():
    raise FileNotFoundError(
        "bike.csv が見つかりません。bike.ipynb と同じフォルダに置いてから実行してください。"
    )

事件一覧 = []

# 千葉県警のCSVの文字コードは CP932 です。
with ファイル名.open("r", encoding="cp932", newline="") as ファイル:
    読み込み = csv.DictReader(ファイル)
    for 行 in 読み込み:
        if 行.get("手口") == "自転車盗":
            事件一覧.append(行)

総件数 = len(事件一覧)
print("読み込み成功：", f"{総件数:,}", "件")
print("最初の記録の市区町村：", 事件一覧[0]["市区町村（発生地）"] if 事件一覧 else "なし")
print("使用した項目：市区町村（発生地）／施錠関係／被害者の年齢／発生場所")

if 総件数 == 0:
    raise ValueError("自転車盗の記録が見つかりませんでした。")

読み込み成功： 10,054 件
最初の記録の市区町村： 千葉市中央区
使用した項目：市区町村（発生地）／施錠関係／被害者の年齢／発生場所


In [5]:
# 市区町村・施錠状況・年代・場所をそれぞれ数える
市区町村別 = {}
施錠別 = {}
年代別 = {}
場所別 = {}

for 事件 in 事件一覧:
    市区町村 = 事件["市区町村（発生地）"]
    施錠状況 = 事件["施錠関係"]
    年代 = 事件["被害者の年齢"]
    場所 = 事件["発生場所"]

    市区町村別[市区町村] = 市区町村別.get(市区町村, 0) + 1
    施錠別[施錠状況] = 施錠別.get(施錠状況, 0) + 1
    年代別[年代] = 年代別.get(年代, 0) + 1
    場所別[場所] = 場所別.get(場所, 0) + 1

# 件数が多い順に並び替え、上位10地域を表示
地域ランキング = sorted(市区町村別.items(), key=lambda データ: データ[1], reverse=True)

print("【自転車盗難件数・市区町村ランキング TOP 10】")
for 順位, (地域, 件数) in enumerate(地域ランキング[:10], start=1):
    print(f"{順位:2}位  {地域:<9}  {件数:>5,}件")

無施錠 = 施錠別.get("施錠せず", 0)
施錠あり = 施錠別.get("施錠した", 0)
十代 = 年代別.get("10歳代", 0)
二十代 = 年代別.get("20歳代", 0)

print("\n【施錠状況】")
print(f"施錠せず：{無施錠:,}件（{無施錠 / 総件数 * 100:.1f}%）")
print(f"施錠した：{施錠あり:,}件（{施錠あり / 総件数 * 100:.1f}%）")
print("\n【若い世代の被害】")
print(f"10歳代：{十代:,}件、20歳代：{二十代:,}件")
print(f"合計：{十代 + 二十代:,}件（{(十代 + 二十代) / 総件数 * 100:.1f}%）")

assert sum(市区町村別.values()) == 総件数
assert sum(施錠別.values()) == 総件数
assert sum(年代別.values()) == 総件数

【自転車盗難件数・市区町村ランキング TOP 10】
 1位  船橋市        1,299件
 2位  市川市          871件
 3位  松戸市          725件
 4位  千葉市中央区       708件
 5位  柏市           564件
 6位  習志野市         528件
 7位  八千代市         455件
 8位  浦安市          407件
 9位  千葉市美浜区       370件
10位  千葉市稲毛区       358件

【施錠状況】
施錠せず：6,879件（68.4%）
施錠した：3,175件（31.6%）

【若い世代の被害】
10歳代：4,460件、20歳代：2,615件
合計：7,075件（70.4%）


In [6]:
# 図を表示するための関数（追加インストール不要）
def 棒グラフを表示(タイトル, データ, 全体件数, 色="#2878B5", 最大件数=None):
    if 最大件数 is None:
        最大件数 = max((件数 for _, 件数 in データ), default=1)

    内容 = [
        '<div style="font-family: sans-serif; max-width:900px; padding:20px; '
        'border:1px solid #d8dee9; border-radius:14px; margin:14px 0;">',
        f'<h3 style="margin-top:0;">{escape(タイトル)}</h3>',
    ]

    for 名前, 件数 in データ:
        幅 = (件数 / 最大件数 * 100) if 最大件数 else 0
        割合 = 件数 / 全体件数 * 100 if 全体件数 else 0
        内容.append(f"""
        <div style="margin:14px 0;">
          <div style="display:flex; justify-content:space-between; gap:16px; font-size:14px;">
            <strong>{escape(str(名前))}</strong>
            <span>{件数:,}件（全体の{割合:.1f}%）</span>
          </div>
          <div style="height:17px;background:#e8eef5;border-radius:8px;margin-top:5px;">
            <div style="height:17px;width:{幅:.1f}%;background:{色};border-radius:8px;"></div>
          </div>
        </div>""")

    内容.append('</div>')
    display(HTML("".join(内容)))

# 図1：市区町村の上位10地域
棒グラフを表示(
    "図1．千葉県内の市区町村別 自転車盗難記録 TOP 10",
    地域ランキング[:10],
    総件数,
    色="#2878B5"
)

In [7]:
# 図2：鍵をかけていたかどうか
棒グラフを表示(
    "図2．盗難に遭った自転車の施錠状況",
    [("施錠せず", 無施錠), ("施錠した", 施錠あり)],
    総件数,
    色="#DB7A34"
)

# 図3：被害者の年代（数の多い上位5項目）
年代ランキング = sorted(年代別.items(), key=lambda データ: データ[1], reverse=True)
棒グラフを表示(
    "図3．被害者の年齢の内訳（上位5項目）",
    年代ランキング[:5],
    総件数,
    色="#26927B"
)

4．数値から分かったこと（発見2つ）
発見①：無施錠の被害が多い
10,054件中、無施錠は6,879件（68.4%）、施錠ありは3,175件（31.6%）でした。
発見②：若い年代の被害が多い
10歳代は4,460件（44.4%）、20歳代は2,615件（26.0%）。合計**7,075件（70.4%）**でした。
地域別ランキング
船橋市が1,299件、市川市が871件、松戸市が725件でした。※件数だけで地域の危険度は判断できません。


5．防犯の提案
学校や駅で「短時間でも施錠・できればツーロック」を呼びかけます。

無施錠の被害は6,879件、10～20歳代の被害は7,075件でした。若い世代にも、駐輪時の施錠を意識してもらうためです。

In [3]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
# 以前制作した「調査機能拡張版チャットボット」のコードを使用
import csv
import re
from pathlib import Path
from datetime import datetime
from collections import Counter
from html import escape
from urllib.parse import quote_plus
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
from IPython.display import HTML, display

# 2025年の「bike.csv」と「000074746.csv」は内容が同一なので一度だけ集計する
資料ファイル = {
    2018: "000047051.csv",
    2019: "000047043.csv",
    2020: "000047035.csv",
    2021: "000062558.csv",
    2022: "000074760.csv",
    2023: "000074753.csv",
    2024: "000075653.csv",
    2025: "bike.csv",
}

公式URL = {
    "データ": "https://www.police.pref.chiba.jp/seisoka/safe-life_publicspace-statistics_00002.html",
    "防犯": "https://www.police.pref.chiba.jp/seisoka/safe-life_publicspace-vehicle_bicycle_theft_00002.html",
    "ツーロック": "https://www.police.pref.chiba.jp/police_department/urayasu/crime-prevention_detail.html",
    "防犯登録": "https://www.police.pref.chiba.jp/seisoka/safe-life_publicspace-vehicle_bicycle_theft_00003.html",
    "警察相談": "https://www.police.pref.chiba.jp/kotsusosaka/window_consultation-trouble_02.html",
}

def CSVを探す(ファイル名):
    # NotebookとCSVを同じ場所に置くのが基本。data フォルダにも対応。
    候補 = [Path.cwd() / ファイル名, Path.cwd() / "data" / ファイル名]
    for パス in 候補:
        if パス.is_file():
            return パス
    return None

def 日付を読む(文字):
    値 = str(文字 or "").strip()
    if not 値 or 値 in ("不明", "不詳"):
        return None
    for 書式 in ("%Y-%m-%d", "%Y/%m/%d", "%Y%m%d"):
        try:
            return datetime.strptime(値, 書式).date()
        except ValueError:
            pass
    return None

def 時を読む(文字):
    値 = str(文字 or "").strip()
    if 値.isdigit() and 0 <= int(値) <= 23:
        return int(値)
    return None

def CSVを読む(パス):
    for 文字コード in ("utf-8-sig", "cp932"):
        try:
            with パス.open(encoding=文字コード, newline="") as f:
                return list(csv.DictReader(f))
        except UnicodeError:
            continue
    raise ValueError(f"文字コードが読み込めません: {パス.name}")

事件一覧 = []
読み込み状況 = []
曜日名 = ("月", "火", "水", "木", "金", "土", "日")

for 資料年, ファイル名 in 資料ファイル.items():
    パス = CSVを探す(ファイル名)
    if パス is None:
        読み込み状況.append(f"⚠ {ファイル名} が見つかりません")
        continue
    行一覧 = CSVを読む(パス)
    追加件数 = 0
    for 行 in 行一覧:
        if 行.get("手口") != "自転車盗":
            continue  # 自動車盗などは集計しない
        日付 = 日付を読む(行.get("発生年月日（始期）"))
        事件一覧.append({
            "資料年": 資料年,
            "市区町村": (行.get("市区町村（発生地）") or "").strip(),
            "町丁目": (行.get("町丁目（発生地）") or "").strip(),
            "警察署": (行.get("管轄警察署（発生地）") or "").strip(),
            "交番": (行.get("管轄交番・駐在所（発生地）") or "").strip(),
            "発生場所": (行.get("発生場所") or 行.get("発生場所の属性") or "").strip(),
            "発生場所詳細": (行.get("発生場所の詳細") or "").strip(),
            "施錠": (行.get("施錠関係") or "").strip(),
            "年齢": (行.get("被害者の年齢") or "").strip(),
            "職業": (行.get("被害者の職業") or "").strip(),
            "日付": 日付,
            "月": 日付.month if 日付 else None,
            "曜日": 曜日名[日付.weekday()] if 日付 else None,
            "時": 時を読む(行.get("発生時（始期）")),
        })
        追加件数 += 1
    読み込み状況.append(f"{資料年}年資料：{追加件数:,}件")

市区町村一覧 = sorted({行["市区町村"] for 行 in 事件一覧 if 行["市区町村"]})
読み込み年一覧 = sorted({行["資料年"] for 行 in 事件一覧})
print("🚲 CSV読み込み結果")
for 状況 in 読み込み状況:
    print(" ・", 状況)
print(f"合計：{len(事件一覧):,}件")
assert 事件一覧, "CSVを読み込めませんでした。Notebookと8つのCSVを同じフォルダに置いてください。"

# ==============================================
# 質問の意味を調べる仕組み（CSVから実際に集計）
# ==============================================

def 判定単語を含む(文章, 言葉):
    return any(単語 in 文章 for 単語 in 言葉)

def 地域を抽出(質問):
    # 「千葉市」は各区をまとめて集計。その他は正式な市区町村名で検索。
    見つけた = [名前 for 名前 in 市区町村一覧 if 名前 in 質問]
    if "千葉市" in 質問 and not any(名前.startswith("千葉市") for 名前 in 見つけた):
        見つけた.append("千葉市")
    # 柏市・松戸市など、地域比較にも対応
    return sorted(set(見つけた), key=lambda x: 質問.index(x))

def 年を抽出(質問):
    年 = re.findall(r"20\d{2}", 質問)
    return [int(x) for x in 年 if int(x) in 読み込み年一覧]

def 対象を絞る(地域="千葉県全体", 資料年="全期間"):
    集合 = 事件一覧
    if 地域 not in ("千葉県全体", "千葉県", ""):
        if 地域 == "千葉市":
            集合 = [行 for 行 in 集合 if 行["市区町村"].startswith("千葉市")]
        else:
            集合 = [行 for 行 in 集合 if 行["市区町村"] == 地域]
    if isinstance(資料年, int):
        集合 = [行 for 行 in 集合 if 行["資料年"] == 資料年]
    return 集合

def 数を集計(データ, 項目):
    無効 = {"", "不明", "不詳", "なし", "None"}
    return Counter(行[項目] for 行 in データ if 行.get(項目) is not None and str(行[項目]).strip() not in 無効)

def 順位表(集計, 単位="件", 最大=7, 並び順="件数"):
    if not 集計:
        return "集計できるデータがありません。"
    項目 = list(集計.items())
    if 並び順 == "名前":
        項目.sort(key=lambda x: str(x[0]))
    else:
        項目.sort(key=lambda x: x[1], reverse=True)
    最大件数 = max(集計.values())
    行一覧 = []
    for 順, (名前, 件数) in enumerate(項目[:最大], 1):
        棒 = "■" * max(1, round(12 * 件数 / 最大件数))
        行一覧.append(f"{順:>2}. {str(名前):<13} {件数:>6,}{単位}  {棒}")
    return "\n".join(行一覧)

def 期間名(資料年):
    return f"{資料年}年資料" if isinstance(資料年, int) else "2018～2025年の資料（読み込めた年のみ）"

def 防犯の要点(データ):
    if not データ:
        return "この条件では記録が見つかりません。"
    鍵 = 数を集計(データ, "施錠")
    場所 = 数を集計(データ, "発生場所")
    時間 = 数を集計(データ, "時")
    文章 = []
    施錠済 = 鍵.get("施錠した", 0)
    無施錠 = 鍵.get("施錠せず", 0)
    if 施錠済 + 無施錠:
        割合 = 無施錠 / (施錠済 + 無施錠) * 100
        文章.append(f"・無施錠だった被害：{無施錠:,}件（施錠状況が分かる被害の{割合:.1f}%）。短時間でも施錠し、ツーロックを心がけましょう。")
    if 場所:
        場所の候補 = [(名前, 件数) for 名前, 件数 in 場所.most_common() if 名前 != "その他"]
        名前, 件数 = 場所の候補[0] if 場所の候補 else 場所.most_common(1)[0]
        文章.append(f"・『その他』を除く最多の場所分類：{名前}（{件数:,}件）。自宅や駐輪場でも油断せず、必ず施錠しましょう。")
    if 時間:
        時, 件数 = 時間.most_common(1)[0]
        文章.append(f"・記録上の最多の発生時（始期）：{時}時台（{件数:,}件）。この値は正確な犯行時刻とは限らないので、時間にかかわらず対策しましょう。")
    return "\n".join(文章) + "\n※被害事例の内訳であり、盗難に遭う確率や対策の効果を示すものではありません。"

# 公式情報と、説明の基礎になる防犯メモ
対策辞典 = {
    "鍵": ("鍵・ツーロック", "・基本は短時間でも施錠。できれば種類の異なる鍵を2つ使用します。\n・車輪だけでなく車体も動かないよう固定し、利用する場所のルールを守りましょう。\n・施錠済みでも盗難記録があるため、鍵だけに頼らず駐輪場所にも注意します。", "ツーロック"),
    "駐輪場": ("駐輪場・駅・買い物", "・管理人や防犯カメラがある、明るく人目につきやすい駐輪場を選びましょう。\n・駅や店舗での短時間の駐輪でも施錠しましょう。\n・CSVには駅名が記録されていないため、特定の駅の盗難件数は分かりません。", "ツーロック"),
    "自宅": ("自宅・マンション", "・自宅の敷地内でも必ず施錠します。\n・可能なら鍵を2つ使い、見通しや管理のよい場所に保管します。\n・マンション共用駐輪場でも防犯対策が必要です。", "ツーロック"),
    "学校": ("学校・通学", "・授業中の長時間駐輪でもツーロックをしましょう。\n・指定された駐輪場を利用し、防犯登録番号と車体番号を控えましょう。\n・被害者の職業の「高校生」「大学生」は、学校で盗まれたことを意味しません。", "ツーロック"),
    "電動": ("電動アシスト自転車・バッテリー", "・駐輪時には車体を必ず施錠。可能ならバッテリーを外して安全に保管します。\n・バッテリーにワイヤー錠を使う方法や、シリアルナンバーを記録する方法も千葉県警が案内しています。\n・このCSVではバッテリー単体の盗難件数は集計できません。", "防犯"),
    "防犯登録": ("防犯登録", "・自転車の防犯登録は法律上の義務で、盗難防止と被害回復に役立つ制度です。\n・登録控え、登録番号、車体番号、自転車の写真を保管しておきましょう。\n・登録の手続きや更新については千葉県警の案内を確認してください。", "防犯登録"),
    "盗難時": ("盗まれたとき", "・まず別の場所へ移動・撤去されていないか安全な範囲で確認します。\n・最寄りの警察署や交番に盗難を届け出ましょう。防犯登録番号、車体番号、写真、最後に駐輪した場所・時間を整理します。\n・位置情報で見つけても自分だけで取り返そうとせず警察へ相談してください。", "警察相談"),
    "見つけた": ("盗まれた自転車を見つけたとき", "・他人が乗っているなど危険な状況では、追跡や直接の対決をしないでください。\n・見つけた場所や日時を記録し、警察に相談してください。\n・被害届を出している場合は、見つかったことも警察へ連絡しましょう。", "警察相談"),
    "保険": ("盗難保険・補償", "・購入店の盗難補償や保険契約の条件を確認しましょう。\n・補償の対象、免責金額、期間、施錠条件、防犯登録の条件、必要書類は契約ごとに違います。\n・このCSVには盗難被害額や保険適用の情報はありません。", "防犯登録"),
    "GPS": ("GPS・紛失防止タグ", "・GPSや紛失防止タグは位置確認の補助になりますが、施錠の代わりにはなりません。\n・通信や電池、設置場所により位置が分からないことがあります。\n・位置が分かっても自力で取り返さず、警察に相談してください。", "警察相談"),
    "防犯": ("防犯対策の基本", "・①短時間でも必ず施錠 ②可能ならツーロック ③自宅でも施錠\n・④管理された駐輪場を選ぶ ⑤防犯登録の控えと車体番号・写真を残す。\n・地域別の件数・無施錠率・場所を調べると、自分の生活に合う対策を考えられます。", "防犯"),
}

def 対策分類(質問):
    # 順序：具体的な対策を優先
    規則 = [
        ("見つけた", ["見つけた", "発見した", "取り返", "取り戻"]),
        ("盗難時", ["盗まれた", "盗難にあった", "盗難に遭った", "被害届", "盗難届", "届出", "届け出"]),
        ("電動", ["電動", "バッテリー"]),
        ("防犯登録", ["防犯登録", "登録番号", "車体番号"]),
        ("保険", ["保険", "補償"]),
        ("GPS", ["GPS", "gps", "追跡タグ", "エアタグ", "AirTag", "位置情報"]),
        ("学校", ["学校で", "大学で", "高校で", "通学", "授業中"]),
        ("自宅", ["自宅", "マンション", "アパート", "家の前"]),
        ("駐輪場", ["駅で", "駅前", "駐輪場で", "買い物", "スーパー", "コンビニ"]),
        ("鍵", ["鍵", "カギ", "ツーロック", "ダブルロック", "ワイヤー錠", "U字ロック", "施錠"]),
        ("防犯", ["対策", "防ぐ", "防犯", "予防", "気を付け", "注意", "おすすめ"]),
    ]
    for 分類, 言葉 in 規則:
        if 判定単語を含む(質問, 言葉):
            return 分類
    return None

def 質問の種類(質問):
    # 件数、年別推移、月別、地域比較などを最優先して分類
    if 判定単語を含む(質問, ["地域比較", "市を比較", "都市を比較", "どっち", "どちら", "VS", "vs"]):return "比較"
    if 判定単語を含む(質問, ["年別", "推移", "経年", "年ごと", "年ごとに", "各年"]):return "年別"
    if 判定単語を含む(質問, ["町丁目", "町名", "地区別", "細かい場所"]):return "町丁目"
    if 判定単語を含む(質問, ["警察署", "管轄", "交番"]):return "警察署"
    if 判定単語を含む(質問, ["ランキング", "多い市", "多い地域", "市区町村別", "上位の地域"]):return "市区町村"
    if 判定単語を含む(質問, ["曜日", "何曜日", "週末", "土日"]):return "曜日"
    if 判定単語を含む(質問, ["何月", "月別", "月ごと", "季節"]):return "月"
    if 判定単語を含む(質問, ["何時", "時間帯", "時刻", "夜間", "朝方", "昼間"]):return "時"
    if 判定単語を含む(質問, ["年齢", "年代", "10代", "20代", "30代", "高齢者", "若い人"]):return "年齢"
    if 判定単語を含む(質問, ["職業", "大学生", "高校生", "学生の被害", "学生は"]):return "職業"
    if 判定単語を含む(質問, ["施錠率", "無施錠", "施錠状況", "鍵なし", "鍵あり", "鍵をかけ"]):return "施錠"
    if 判定単語を含む(質問, ["発生場所", "場所", "どこで", "住宅で", "駐輪場の件数"]):return "場所"
    if 判定単語を含む(質問, ["件数", "何件", "総数", "何台", "盗難数"]):return "件数"
    if 判定単語を含む(質問, ["まとめ", "特徴", "傾向", "概要", "詳しく", "教えて", "知りたい"]):return "概要"
    return None

def 回答を作る(質問, 選択地域="千葉県全体", 選択年="全期間"):
    質問 = 質問.strip()
    if not 質問:
        return "質問を入力してください。", "防犯"

    地域リスト = 地域を抽出(質問)
    年リスト = 年を抽出(質問)
    地域 = 地域リスト[0] if 地域リスト else 選択地域
    分析項目 = 質問の種類(質問)
    対策項目 = 対策分類(質問)
    # 「何件盗まれた？」は統計の質問であり、被害届の相談ではない
    if 分析項目 is not None and 対策項目 == "盗難時" and 判定単語を含む(質問, ["何件", "件数", "統計", "割合"]):
        対策項目 = None
    # 防犯登録や保険の「方法を教えて」などは、関係ない全件数を表示しない
    if 分析項目 == "概要" and 対策項目 is not None and not 判定単語を含む(質問, ["データ", "統計", "傾向", "特徴", "地域", "集計", "割合"]):
        分析項目 = None
    # 県内ランキングは、画面で市を選んでいても原則として千葉県全体で順位付け
    if 分析項目 == "市区町村" and not 地域リスト:
        地域 = "千葉県全体"
    # 年別・比較などでは年を指定していても系列比較しやすいよう全期間を使用
    資料年 = (年リスト[0] if 年リスト else 選択年)
    if 分析項目 == "年別":
        資料年 = "全期間"
    データ = 対象を絞る(地域, 資料年)
    表題 = f"【{地域}／{期間名(資料年)}】\n登録件数：{len(データ):,}件"
    出力 = []
    if 分析項目 in ("比較", "件数", "概要", "年別", "市区町村", "町丁目", "警察署", "曜日", "月", "時", "場所", "年齢", "職業", "施錠") or not 対策項目:
        出力.append(表題)
    if not データ and 分析項目 != "年別":
        return "指定した条件のデータは見つかりませんでした。地域や資料年を変更してください。", "データ"

    if 分析項目 == "比較":
        地域2 = 地域リスト[1] if len(地域リスト) > 1 else None
        if 地域2 is None:
            return "比較する2地域を質問に入力してください。例：『柏市と松戸市を比較』", "データ"
        D1, D2 = 対象を絞る(地域, 資料年), 対象を絞る(地域2, 資料年)
        def 無施錠率(行一覧):
            鍵=数を集計(行一覧,"施錠"); 分母=鍵.get("施錠せず",0)+鍵.get("施錠した",0)
            return f'{鍵.get("施錠せず",0)/分母*100:.1f}%' if 分母 else "不明"
        return (f"【{地域} と {地域2} の比較／{期間名(資料年)}】\n"
                f"{地域}：{len(D1):,}件／被害の無施錠割合 {無施錠率(D1)}\n"
                f"{地域2}：{len(D2):,}件／被害の無施錠割合 {無施錠率(D2)}\n"
                f"差（{地域}－{地域2}）：{len(D1)-len(D2):+,}件\n\n"
                "※市区町村ごとの人口・自転車台数が異なるため、件数だけで危険度を比較できません。"), "データ"

    if 分析項目 == "年別":
        年別 = Counter(行["資料年"] for 行 in 対象を絞る(地域, "全期間"))
        出力.append("【資料年別の件数】\n"+ "\n".join(f"{年}年資料：{年別.get(年,0):,}件" for 年 in 読み込み年一覧))
        出力.append("※この表は『資料年別』の集計です。発生年月日ベースで集計した結果とは異なることがあります。")
    elif 分析項目 == "市区町村":
        出力.append("【市区町村別・上位10地域】\n"+順位表(数を集計(データ,"市区町村"),最大=10))
    elif 分析項目 == "町丁目":
        出力.append("【町丁目別・上位10】\n"+順位表(数を集計(データ,"町丁目"),最大=10))
    elif 分析項目 == "警察署":
        種類 = "交番" if "交番" in 質問 else "警察署"
        出力.append(f"【{種類}別・上位10】\n"+順位表(数を集計(データ,種類),最大=10))
    elif 分析項目 == "曜日":
        集計 = 数を集計(データ,"曜日")
        出力.append("【曜日別の件数】\n"+ "\n".join(f"{x}曜日：{集計.get(x,0):,}件" for x in ("月","火","水","木","金","土","日")))
        出力.append("※発生年月日（始期）が不明な記録は集計されません。実際の犯行曜日とは限りません。")
    elif 分析項目 == "月":
        集計=数を集計(データ,"月")
        出力.append("【月別の件数】\n" + "\n".join(f"{x:>2}月：{集計.get(x,0):,}件" for x in range(1,13)))
        出力.append("※発生年月日（始期）を使った集計です。発生日が不明な記録は含まれません。")
    elif 分析項目 == "時":
        出力.append("【発生時（始期）が多い時間帯・上位10】\n" + 順位表(Counter({f"{k:02d}時台":v for k,v in 数を集計(データ,"時").items()}),最大=10))
        出力.append("※発生時（始期）は実際の犯行時刻と一致するとは限りません。")
    elif 分析項目 == "場所":
        出力.append("【発生場所の分類・上位10】\n"+順位表(数を集計(データ,"発生場所"),最大=10))
        出力.append("※駅名・施設名の特定はできません。発生場所の分類はCSVの表記です。")
    elif 分析項目 == "年齢":
        集計 = 数を集計(データ,"年齢")
        出力.append("【被害者の年齢区分】\n"+順位表(集計,最大=12))
        for 年代 in ("10代","20代","30代","40代","50代","60代","70代"):
            if 年代 in 質問:
                区分 = 年代.replace("代","歳代")
                出力.append(f"指定した{年代}の被害記録：{集計.get(区分,0):,}件")
                break
    elif 分析項目 == "職業":
        集計 = 数を集計(データ,"職業")
        出力.append("【被害者の職業区分】\n"+順位表(集計,最大=12))
        for 職業 in ("大学生", "高校生", "中学生", "小学生"):
            if 職業 in 質問:
                出力.append(f"指定した{職業}の被害記録：{集計.get(職業,0):,}件")
                break
        出力.append("※『大学生』『高校生』は学校敷地内で発生したことを意味しません。")
    elif 分析項目 == "施錠":
        集計=数を集計(データ,"施錠")
        有効 = 集計.get("施錠せず",0)+集計.get("施錠した",0)
        出力.append("【盗難被害に遭った自転車の施錠状況】\n"
                    +f"施錠せず：{集計.get('施錠せず',0):,}件\n"
                    +f"施錠した：{集計.get('施錠した',0):,}件\n"
                    + (f"無施錠の割合：{集計.get('施錠せず',0)/有効*100:.1f}%" if 有効 else "割合不明"))
        出力.append("※これは盗難被害の内訳です。施錠の有無による盗難確率を示すものではありません。")
    elif 分析項目 == "概要":
        出力.append("【主な傾向と対策】\n"+防犯の要点(データ))

    if 対策項目:
        if 対策項目 == "防犯" and 分析項目 == "件数":
            出力.append("【データから考える防犯上の注意】\n" + 防犯の要点(データ))
        見出し, 内容, 出典 = 対策辞典[対策項目]
        出力.append(f"【{見出し}】\n{内容}")
    elif 分析項目 in ("概要", "件数", None):
        出力.append("【防犯対策（集計結果を参考に）】\n"+防犯の要点(データ))
    elif 分析項目 in ("施錠","場所","時","曜日","月","比較"):
        出力.append("【防犯の基本】\n短時間でも鍵をかけ、可能ならツーロック。自宅でも施錠し、管理された駐輪場を利用しましょう。")

    if 分析項目 is None and 対策項目 is None:
        出力.append("【質問例】\n『柏市の件数』『柏市と松戸市を比較』『年別推移』『月別』『曜日別』『何時に多い？』『町丁目ランキング』『被害者の年代』『鍵の対策』『盗まれたら？』『電動自転車の防犯』")

    出典 = (対策辞典[対策項目][2] if 対策項目 else "防犯")
    return "\n\n".join(出力), 出典


# ==============================================
# 見やすく質問できる画面（自由入力＋テーマボタン）
# ==============================================
if widgets is not None:

    地域の候補 = ["千葉県全体", "千葉市"] + [x for x in 市区町村一覧 if not x.startswith("千葉市")]
    地域選択 = widgets.Dropdown(options=地域の候補, value="千葉県全体",description="地域",layout=widgets.Layout(width="320px"))
    年選択 = widgets.Dropdown(options=["全期間"]+sorted(読み込み年一覧, reverse=True),value=max(読み込み年一覧),description="資料年",layout=widgets.Layout(width="240px"))
    質問欄 = widgets.Text(placeholder="例：柏市の鍵をかけていなかった割合と対策を教えて",description="質問",layout=widgets.Layout(width="780px"))
    質問ボタン = widgets.Button(description="質問する",button_style="primary",icon="comment",layout=widgets.Layout(width="125px"))
    消去ボタン = widgets.Button(description="履歴を消す",icon="trash",layout=widgets.Layout(width="125px"))
    会話表示 = widgets.Output(layout=widgets.Layout(border="1px solid #d8e0e8",padding="14px",width="100%",max_height="500px",overflow_y="auto"))

    テーマ一覧 = [
        ("件数と防犯", "盗難の件数と対策を教えて"),
        ("地域ランキング", "盗難が多い市区町村ランキング"),
        ("2地域の比較", "柏市と松戸市を比較"),
        ("年別推移", "年別推移"),
        ("町丁目", "町丁目別ランキング"),
        ("警察署", "警察署別ランキング"),
        ("月別", "月別の盗難件数"),
        ("曜日別", "曜日別の盗難件数"),
        ("時間帯", "何時ごろ盗難が多い"),
        ("発生場所", "盗難の発生場所の内訳"),
        ("施錠状況", "施錠状況を教えて"),
        ("被害者の年齢", "被害者の年齢別内訳"),
        ("学生などの職業", "被害者の職業別内訳"),
        ("鍵の対策", "鍵・ツーロックの対策"),
        ("自宅の防犯", "自宅でできる自転車の対策"),
        ("駅・駐輪場", "駅での駐輪場の対策"),
        ("学校・通学", "通学中に気を付けたいこと"),
        ("電動・バッテリー", "電動アシスト自転車のバッテリー対策"),
        ("防犯登録", "防犯登録の方法を知りたい"),
        ("盗まれたら", "自転車が盗まれたらどうする"),
        ("見つけたら", "盗まれた自転車を見つけたら"),
        ("保険", "盗難保険について知りたい"),
        ("GPS", "GPSで盗難対策できる？"),
    ]

    def 返信を表示(質問):
        答え, 出典種 = 回答を作る(質問,地域選択.value,年選択.value)
        # ユーザー入力をそのままHTMLに埋め込まない（安全のためエスケープ）
        質問HTML = escape(質問)
        答えHTML = escape(答え).replace("\n", "<br>")
        URL = 公式URL.get(出典種, 公式URL["防犯"])
        検索語 = f"site:police.pref.chiba.jp 自転車盗 {質問}"
        検索URL = "https://www.google.com/search?q=" + quote_plus(検索語)
        with 会話表示:
            display(HTML(f"""<div style="font-family:'Yu Gothic',sans-serif;font-size:14px;line-height:1.7;padding:8px 0;">
            <div style="margin:8px 0 12px auto;padding:9px 12px;border-radius:11px;background:#e9f3ff;max-width:85%;width:fit-content;">🙋 {質問HTML}</div>
            <div style="padding:12px 14px;background:#f4f7f5;border-left:4px solid #32906a;border-radius:8px;white-space:normal;">🚲 <b>チャットボット</b><br><br>{答えHTML}<br><br>
            <a href="{escape(URL,quote=True)}" target="_blank" rel="noopener noreferrer">📖 千葉県警の関連情報を開く</a>
            <span style="margin:0 8px;color:#aaa;">|</span>
            <a href="{escape(検索URL,quote=True)}" target="_blank" rel="noopener noreferrer">🔎 公式サイト内をさらに検索</a></div></div>"""))

    def 送信する(_=None):
        質問 = 質問欄.value.strip()
        if 質問:
            返信を表示(質問)
            質問欄.value = ""

    def 履歴を消す(_=None):
        会話表示.clear_output()

    質問ボタン.on_click(送信する)
    消去ボタン.on_click(履歴を消す)

    テーマボタン = []
    for 見出し, 例文 in テーマ一覧:
        b = widgets.Button(description=見出し,layout=widgets.Layout(width="151px",height="34px"),style={"font_weight":"normal"})
        b.on_click(lambda _, 例文=例文: 返信を表示(例文))
        テーマボタン.append(b)

    print("\n🚲 自転車盗難・防犯リサーチBot（拡張版）")
    print("・地域と資料年を選ぶ → テーマボタンを押す、または質問を入力")
    print("・地域名・資料年を質問文に入れた場合は質問文を優先")
    print("・2都市を比較するときは『柏市と松戸市を比較』と入力")

    display(widgets.VBox([
        widgets.HTML('<h3 style="margin:6px 0;">🚲 自転車盗難・防犯リサーチチャットボット</h3><p>CSVから地域別の傾向を調べ、関連する対策を確認できます。</p>'),
        widgets.HBox([地域選択,年選択]),
        widgets.HBox([質問欄,質問ボタン]),
        widgets.HTML('<b>テーマから探す</b>'),
        widgets.GridBox(テーマボタン,layout=widgets.Layout(grid_template_columns="repeat(5, 151px)",grid_gap="6px 6px")),
        消去ボタン,
        会話表示,
    ]))
    返信を表示("盗難の件数と対策を教えて")
else:
    print("\n【チャットボットの回答例】（ボタン操作にはipywidgetsが必要です）")
    for 例文 in ("2025年の柏市の盗難件数と対策を教えて", "柏市と松戸市を比較", "鍵・ツーロックの対策"):
        答え, _ = 回答を作る(例文, "千葉県全体", 2025)
        print("\n質問：" + 例文 + "\n回答：" + 答え[:550])
    print("\nチャット画面を表示するには、別セルで %pip install ipywidgets を実行してください。")


🚲 CSV読み込み結果
 ・ 2018年資料：10,989件
 ・ 2019年資料：10,215件
 ・ 2020年資料：7,190件
 ・ 2021年資料：6,317件
 ・ 2022年資料：7,346件
 ・ 2023年資料：9,439件
 ・ 2024年資料：9,943件
 ・ 2025年資料：10,054件
合計：71,493件

🚲 自転車盗難・防犯リサーチBot（拡張版）
・地域と資料年を選ぶ → テーマボタンを押す、または質問を入力
・地域名・資料年を質問文に入れた場合は質問文を優先
・2都市を比較するときは『柏市と松戸市を比較』と入力


7．まとめ
bike.csv の10,054件を集計しました。無施錠は68.4%、10～20歳代は**70.4%で、最多の市区町村は船橋市（1,299件）**でした。

この結果から、学校や駅での施錠・ツーロックの呼びかけを提案しました。以前のチャットボットも使い、地域別の件数や防犯対策を調べられるようにしました。